Start building interruption table with id and text column 

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold AS
SELECT
  text
FROM beneco_pipeline.silver.interruption_silver;
SELECT * FROM beneco_pipeline.gold.interruption_gold;

### Type Capturing
* Capture interruption type from power service advisories and append to table. 
* Create new silver table only including text and type
* SELECT query to check if any advisories were not categorized into any type through the regexp_extract

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold
SELECT
    text,
    regexp_extract(text,r'(\w+)\s+Power Interruption', 1) AS type
FROM beneco_pipeline.gold.interruption_gold;
select * from beneco_pipeline.gold.interruption_gold;

### Start datetime capturing
* Capture start date from advisory text using regex
* Concatinate time of outage with date of outage using regexp_extract
* Extraction will differ depending on advisory format


In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold
AS SELECT
    text,
    type,
    date_format(try_to_timestamp(
    CASE 
        -- for DATE: weekday, mmmm, dd, yyyy
        WHEN text RLIKE r'(?i)DATE:\s+\w+\s?,?\s+(\w+\s+\d{1,2},\s+\d{4})' THEN (
            regexp_extract(text,r'DATE:\s+\w+\s?,?\s+(\w+\s+\d{1,2},\s+\d{4})',1) ||' '|| -- Date
            regexp_extract(text,r'(?i)START(?:ED)?\s?:(?:\s+)?(\d{1,2}:\s?\d{2}\s?[AP]M)',1) || --time START(ED): XX:XX AM/PM
            regexp_extract(text,r'(?i)START(?:ED)?:(?:\s+)?\w+\s?,?\s+\w+\s+\d{1,2},\s+\d{4}\s[;aAtT]\s(\d{1,2}:\d{2}\s?[AP]M)',1) || -- for case: START(ED): (day), MMMM dd, yyyy at XX:XX AM/PM
            regexp_extract(text,r'(?i)TIME\s?:\s?(\d{1,2}:\d{2}\s?[AP]M)',1) -- for case TIME: HH:MM AM/PM
            -- ||' 1'
        )
        WHEN TEXT RLIKE r'\s?:\s+\w+,?\s?,?\s+(\w+\s+\d{1,2},\s+\d{4})\s+[aA][tT](\s+\d+:\d+\s+\w+)' THEN 
            regexp_extract(text,r'(?i)STARTED\s?:\s+\w+\s?,?\s+(\w+\s+\d{1,2},\s+\d{4})\s+[aA][tT]\s+?\d+:\d+\s+\w+',1)||' '|| -- date 
            regexp_extract(text,r'(?i)STARTED\s?:\s+\w+\s?,?\s+\w+\s+\d{1,2},\s+\d{4}\s+[aA][tT]\s+?(\d+:\d+\s+\w+)',1) -- time
            -- ||' 2'
        ELSE 'nope'
    END,'MMMM d, yyyy h:mm a'), 'yyyy-MM-dd HH:mm')
    AS start
    
FROM beneco_pipeline.gold.interruption_gold;

SELECT * 
FROM beneco_pipeline.gold.interruption_gold 
where start is null;

### End datetime capturing
* Capture end date from advisory text using regex
* Concatinate time of outage with date of outage using regexp_extract
* Extraction will differ depending on advisory format


In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold
AS SELECT
    text, 
    type,
    start,
    date_format(try_to_timestamp(
    CASE 
        WHEN text RLIKE r'DATE AND TIME RESTORED' THEN(
            regexp_extract(text,r'RESTORED:(?: \w+,)? (\w+ \d{1,2}, \d{4});? \d{1,2}:\d{2} [AP]M',1) ||' '|| -- Date
            regexp_extract(text,r'RESTORED:(?: \w+,)? \w+ \d{1,2}, \d{4};? (\d{1,2}:\d{2} [AP]M)',1) -- Time
        )
        WHEN text RLIKE r'DATE:\s+\w+\s?,?\s+(\w+\s+\d{1,2},\s+\d{4})' THEN (
            regexp_extract(text,r'DATE:\s+\w+\s?,?\s+(\w+\s+\d{1,2},\s+\d{4})',1) ||' '|| -- Date
            CASE
                WHEN text RLIKE r'(?:RESTORED?|ENERGIZED)\s?:(?:\s+)?(\d{1,2}:\s?\d{2}\s?[AP]M)' THEN (
                regexp_extract(text,r'(?:RESTORED?|ENERGIZED)\s?:(?:\s+)?(\d{1,2}:\s?\d{2}\s?[AP]M)',1) --time RESTORED: XX:XX AM/PM
                )
                WHEN text RLIKE r'(?:RESTORED?|ENERGIZED):?(?:\s+)?\w+\s?,?\s+\w+\s+\d{1,2},\s+\d{4}\s[;aAtT]\s(\d{1,2}:\d{2}\s?[aApP][mM])' THEN (
                regexp_extract(text,r'(?:RESTORED?|ENERGIZED):?(?:\s+)?\w+\s?,?\s+\w+\s+\d{1,2},\s+\d{4}\s[;aAtT]\s(\d{1,2}:\d{2}\s?[aApP][mM])',1) -- for case: RESTORED/ENERGIZED: (day), MMMM dd, yyyy at XX:XX AM/PM
                )
                WHEN text RLIKE r'(TO|to)\s?(\d{1,2}:\d{2}\s?[AP]M)' THEN (
                regexp_extract(text,r'(?:TO|to)\s(\d{1,2}:\d{2}\s?[AP]M)',1) -- for case TIME: HH:MM AM/PM
                )
                ELSE 'nope'
            END
        )
        WHEN text RLIKE r'\s?:\s+\w+,?\s?,?\s+(\w+\s+\d{1,2},\s+\d{4})\s+[aA][tT](\s+\d+:\s?\d+\s+\w+)' THEN 
            regexp_extract(text,r'RESTORED\s?:\s+\w+\s?,?\s+(\w+\s+\d{1,2},\s+\d{4})\s+[aA][tT]\s+?\d+:\s?\d+\s+\w+',1)||' '|| -- date 
            regexp_extract(text,r'RESTORED\s?:\s+\w+\s?,?\s+\w+\s+\d{1,2},\s+\d{4}\s+[aA][tT]\s+?(\d+:\s?\d+\s+\w+)',1) -- time
        ELSE 'nope'
    END
    , 'MMMM d, yyyy h:mm a'), 'yyyy-MM-dd HH:mm') 
    AS end
    
FROM beneco_pipeline.gold.interruption_gold;

### Remark capturing
* Capture outage cause, status or activity from text column


In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold
AS SELECT
    text, 
    type,
    start,
    end,
    CASE 
      WHEN text RLIKE r'(?i)AND AREAS AFFECTED:?' THEN(
        regexp_extract(text,r'(?i)(AND AREAS AFFECTED:?[^\n\r]+)',1))
      WHEN text RLIKE r'(?i)(PURPOSE):?' THEN(
        regexp_extract(text,r'(?i)(PURPOSE:?[^\n\r]+)',1)
        ||'\n'||regexp_extract(text,r'(?i)(STATUS:?[^\n\r]+)',1))
      WHEN text RLIKE r'(?i)CAUSE:?' THEN(
        regexp_extract(text,r'(?i)(CAUSE[^\n\r]+)',1)
        ||'\n'||regexp_extract(text,r'(?i)(STATUS:?[^\n\r]+)',1))
      WHEN text RLIKE r'(?i)STATUS:?' THEN(
        regexp_extract(text,r'(?i)(STATUS:?[^\n\r]+)',1))
      ELSE 'No remark'
    END
    AS remark
FROM beneco_pipeline.gold.interruption_gold;

-- SELECT text, remark FROM beneco_pipeline.gold.interruption_gold
-- WHERE LOWER(remark) LIKE '%electrocuted%';

In [0]:
%sql
-- Save data to a separate backup table
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_backup AS
SELECT * FROM beneco_pipeline.gold.interruption_gold;

-- Recreate main table with primary key
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  text STRING,
  type STRING,
  start TIMESTAMP,
  end TIMESTAMP,
  remark STRING
);

-- Insert from backup
INSERT INTO beneco_pipeline.gold.interruption_gold (text, type, start, end, remark)
SELECT text, type, start, end, remark FROM beneco_pipeline.gold.interruption_backup;

-- Clean up backup
DROP TABLE beneco_pipeline.gold.interruption_backup;

SELECT * FROM beneco_pipeline.gold.interruption_gold LIMIT 10;

In [0]:
%sql
SELECT * FROM beneco_pipeline.gold.interruption_gold;

Remove temporary text column

In [0]:
%skip
%sql
-- Save data to temp table
CREATE OR REPLACE TEMP VIEW temp_interruption AS
SELECT id, type, start, end, remark
FROM beneco_pipeline.gold.interruption_gold;

CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  type STRING,
  start TIMESTAMP,
  end TIMESTAMP,
  remark STRING
);

-- Insert data with explicit ids
INSERT INTO beneco_pipeline.gold.interruption_gold (id, type, start, end, remark)
SELECT id, type, start, end, remark
FROM temp_interruption;

SELECT * FROM beneco_pipeline.gold.interruption_gold WHERE end IS NULL;